# SQL Worksheet — Week4

Use the following tables from the Riva Data Platform:

- `rivadataplatform.dataproduct.dim_batch`
- `rivadataplatform.dataproduct.dim_class`
- `rivadataplatform.dataproduct.fact_attendance`
- `rivadataplatform.dataproduct.dim_student`
- `rivadataplatform.dataproduct.dim_date`

**Instructions**
- Write SQL for each question.
- Do not modify the source data.
- Use clear aliases where JOINs are involved.
- Unless a question specifically asks for a particular column, select only the columns needed to answer it.


## Tables / Relationships

Useful keys:
- `dim_student.student_key` ↔ `fact_attendance.student_key`
- `dim_class.class_key` ↔ `fact_attendance.class_key`
- `dim_batch.batch_key` ↔ `fact_attendance.batch_key`
- `dim_class.batch_id` ↔ `dim_batch.batch_id`

## Question 1 — Complete Attendance Detail
Join all five tables and return one row per attendance record with student identity, null-safe city, batch, class date, calendar day, topic, class status, attendance status, and remarks. Replace null remarks and topics with labels and sort by date and student.

In [0]:
--Write your code here
select 
s.student_id,
s.student_name,
b.batch_name,
c.class_date,
c.status as class_status,
a.attendance_status,
d.day_name as calendar_day,


coalesce(s.city,'city not available') as city,
coalesce(c.topic, 'topic not assigned') as topic,
coalesce(a.remarks, 'no remarks') as remarks

from rivadataplatform.dataproduct.fact_attendance a
join rivadataplatform.dataproduct.dim_students s 
on a.student_key=s.student_key

join rivadataplatform.dataproduct.dim_class c 
on a.class_key = c.class_key

join rivadataplatform.dataproduct.dim_batch b 
on a.batch_key = b.batch_key

join rivadataplatform.dataproduct.dim_date d 
on a.date_key = d.date_key

order by 
c.class_date,
s.student_name;


## Question 2 — Class Attendance Scorecard
For every class, including classes with no attendance, join class, batch, attendance, and student dimensions. Group by class and return distinct students, Present/Late/Absent counts, and total records. Label null topics and preserve zero counts.

In [0]:
--Write your code here
select 
c.class_id,
c.class_date,
b.batch_name,

coalesce(c.topic, 'topic not assigned') as topic,

c.status as class_status,
count(distinct s.student_key) as distinct_student,
sum(
    case 
        when a.attendance_status = 'Present' then 1
        else 0
        end
) as present_count,
sum(
    case 
        when a.attendance_status = 'Absent' then 1
        else 0
        end
) as absent_count,
sum(
    case 
        when a.attendance_status = 'Late' then 1
        else 0
        end
) as late_count,

count(a.attendance_id) as total_records

from rivadataplatform.dataproduct.dim_class c 



left join rivadataplatform.dataproduct.fact_attendance a 
on a.class_key = c.class_key

left join rivadataplatform.dataproduct.dim_batch b
on c.batch_id = b.batch_id

left join rivadataplatform.dataproduct.dim_students s 
on a.student_key=s.student_key

group by 
c.class_id,
c.class_date,
b.batch_name,
coalesce(c.topic, 'topic not assigned'),
c.status
 order by c.class_date;


## Question 3 — Attendance Rate by Class Date
Join class, batch, attendance, and date dimensions. For each class date and topic, calculate total records, attended records (`Present` + `Late`), absent records, and attendance rate. Use `NULLIF`, label null topics, and return only dates with attendance.

In [0]:
--Write your code here
select 
 c.class_date,
 coalesce(c.topic,'topic not assigned') as topic,
 count(a.attendance_id) as total_records,

 sum(
    case 
        when a.attendance_status in ('present', 'late') then 1
        else 0
        end
        
        
        ) as attendance_records,

        sum(
    case 
        when a.attendance_status = 'absent' then 1
        else 0
        end
        
        ) as absent_records,

        ROUND(
        100.0 *
        SUM(
            CASE
                WHEN a.attendance_status IN ('Present', 'Late') THEN 1
                ELSE 0
            END
        )
        /
        NULLIF(COUNT(a.attendance_id), 0),
        2
    ) AS attendance_rate

    from rivadataplatform.dataproduct.dim_class c  
    join rivadataplatform.dataproduct.dim_batch b  
    on c.batch_id=b.batch_id

    join rivadataplatform.dataproduct.fact_attendance a 
    on a.class_key = c.class_key

    join rivadataplatform.dataproduct.dim_date d  
    on a.date_key = d.date_key
    group by 
    c.class_date,
    coalesce(c.topic,'topic not assigned')

    order by c.class_date;

## Question 4 — Classes With Absence Risk
Join classes, batches, attendance, and students. Group by class, return absent count, distinct affected students, and absent percentage, and keep only classes where the absent count is greater than zero. Include null-safe topic text.

In [0]:
--Write your code here

select 
c.class_id,
c.class_date,
b.batch_name,

coalesce(c.topic, 'topic not assigned') as topic,

sum(case when a.attendance_status = 'Present' then 1 else 0 end) as present_count,
sum(case when a.attendance_status = 'Absent' then 1 else 0 end) as absent_count,

count(distinct case when a.attendance_status = 'Present' then s.student_key end ) as affected_students,
round(100.0* sum (case when a.attendance_status = 'Absent' then 1 else 0 end)/
nullif(count(a.attendance_id),0),2) as absent_percentage

from rivadataplatform.dataproduct.dim_class c

join rivadataplatform.dataproduct.dim_batch b 
on c.batch_id = b.batch_id

join rivadataplatform.dataproduct.fact_attendance a 
on a.class_key = c.class_key

join rivadataplatform.dataproduct.dim_students s
on a.student_key = s.student_key

group by
c.class_id,
c.class_date,
b.batch_name,
coalesce(c.topic, 'topic not assigned')

having sum(case when a.attendance_status = 'Absent' then 1 else 0 end)>0

order by c.class_date;





## Question 5 — Student Issues by Location
Join students, attendance, classes, batches, and dates. Group by student and null-safe city, count Late and Absent records, calculate the issue rate, and return only students with at least one issue. Order by issue rate descending.

In [0]:
--Write your code here

select 
s.student_name,
coalesce (s.city,"City not available") as City,
sum(case when a.attendance_status = 'Late' then 1 else 0 end ) as late_count,
sum(case when a.attendance_status = 'absent' then 1 else 0 end) as absent_count,
count(a.attendance_id) as total_records,
round(100.0* sum(case when a.attendance_status in ('late', 'absent')then 1 else 0 end)/
nullif(count(a.attendance_id),0),2) as issue_rate


from rivadataplatform.dataproduct.fact_attendance a 

join rivadataplatform.dataproduct.dim_students s
on a.student_key = s.student_key

join rivadataplatform.dataproduct.dim_class c
on a.class_key = c.class_key

join rivadataplatform.dataproduct.dim_batch b
on c.batch_id = b.batch_id

join rivadataplatform.dataproduct.dim_date d
on a.date_key = d.date_key

group by 
s.student_key,
s.student_name,
coalesce (s.city,"City not available")

having sum(case when a.attendance_status in ('late', 'absent') Then 1 else 0 end )>0

order by issue_rate desc




## Question 6 — Batch Relationship Data Quality
Join attendance to class and both batch representations. Group by class and recorded/expected batch names, then return only mismatches between the attendance `batch_key` and the batch identified by `dim_class.batch_id`. Include mismatch record counts and null-safe labels.

In [0]:
--Write your code here

select 
c.class_id,
c.class_date,

coalesce(c.topic, 'Topic not assigned') as topic,

coalesce (br.batch_name,'Recorded batch not found') as recorded_batch,

COALESCE(be.batch_name,'Expected batch not found') AS expected_batch,


count (a.attendance_id) as mismatch_records

from rivadataplatform.dataproduct.fact_attendance a 

join rivadataplatform.dataproduct.dim_class c
on a.class_key = c.class_key

left join rivadataplatform.dataproduct.dim_batch br 
on a.batch_key = br.batch_key
left join rivadataplatform.dataproduct.dim_batch be
on c.batch_id = be.batch_id

where not (a.batch_key <=> be.batch_key)


GROUP BY
    c.class_id,
    c.class_date,
    COALESCE(c.topic, 'Topic not assigned'),
    COALESCE(br.batch_name, 'Recorded batch not found'),
    COALESCE(be.batch_name, 'Expected batch not found')

ORDER BY
    c.class_date;

